# Bar Path Model

### Training a model for bar path tracking

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Grab data from AirTable

In [2]:
from ast import literal_eval
from pyairtable import Api as airtable_api

In [3]:
DATA_COLUMNS = ['AccX', 'AccY', 'AccZ', 'VelX', 'VelY', 'VelZ', 'PosX', 'PosY', 'PosZ', 'Pitch', 'Roll', 'Yaw']
EXERCISE_TYPES = ['Ab Rollout', 'Bench', 'DB Bench', 'DB Bicep Curl', 'DB Deadlift', 
              'DB Goblet Squat', 'DB Side Raise', 'Deadlift', 'Front Squat', 
              'Horizontal Chest Press', 'Lat Pulldown', 'OHP', 'Row', 
              'Seated Cable Rows', 'Squat', 'Tricep Cable Pushdown']

CUTOFF_FREQ = 0.1
SAMPLE_RATE = 100

WINDOW_SIZE = 100
STEP_SIZE = 50

AIR_TABLE_API_KEY = 'patMswkrKzfHWSG3U.a882cccc6e7709b3a24b42b7f9c0ccd48ed15614f13c54e4d3dbaa55ba8ffffb'
BASE_ID = 'appaCiWbsAFmlkOLF'
TABLE_NAME = 'tblZtbDVHfCaoTnsH'
BASE_URL = 'https://api.airtable.com/v0/'

EXERCISE = 'Bench'
VALID_DATA = 'true'
MIN_REPS = '1'

FORMULA = "AND({ValidData} = 'true', " + \
        "{StartRepTime} != '', " + \
        "{EndRepTime} != '', " + \
        "{Reps} >= '" + MIN_REPS + "')"

FORMULA

"AND({ValidData} = 'true', {StartRepTime} != '', {EndRepTime} != '', {Reps} >= '1')"

In [4]:
def to_numpy(data):
    return np.array(literal_eval(data))

def to_float(data):
    return float(data)

In [5]:
api = airtable_api(AIR_TABLE_API_KEY)
table = api.table(BASE_ID, TABLE_NAME)

matches = table.all(formula=FORMULA)
print("Number of matching records: ", len(matches))

Number of matching records:  202


In [6]:
df = pd.DataFrame()

data_dict = []

for match in matches:
    data = match['fields']
    
    data_dict.append({
        'Date': data.get('Date'),
        'Exercise': data.get('Exercise'),
        'Lifter': data.get('Lifter'),
        'WorkoutTime': data.get('WorkoutTime'),
        'Reps': data.get('Reps'),
        'Weight': data.get('Weight'),
        'Intensity': data.get('Intensity'),
        'Notes': data.get('Notes'),
        'StartRepTime': data.get('StartRepTime'),
        'EndRepTime': data.get('EndRepTime'),
        'Counter': to_numpy(data.get('Counter')),
        'TimeBetweenSamples': to_numpy(data.get('TimeBetweenSamples')),
        'AccX': to_numpy(data.get('AccX')),
        'AccY': to_numpy(data.get('AccY')),
        'AccZ': to_numpy(data.get('AccZ')),
        'Pitch': to_numpy(data.get('Pitch')),
        'Roll': to_numpy(data.get('Roll')),
        'Yaw': to_numpy(data.get('Yaw')),
        'HeartRate': to_numpy(data.get('HeartRate')),
    })

df = pd.concat([pd.DataFrame([d]) for d in data_dict], ignore_index=True)
df

,Date,Exercise,Lifter,WorkoutTime,Reps,Weight,Intensity,Notes,StartRepTime,EndRepTime,Counter,TimeBetweenSamples,AccX,AccY,AccZ,Pitch,Roll,Yaw,HeartRate
0,"Apr 16, 2024 at 4:45:01 PM",Horizontal Chest Press,Anwar,28.69,8,145,10,None,6.72,25.34,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[222003, 10, 9, 10, 10, 10, 12, 7, 10, 10, 10,...","[0.254, 0.409, -0.13, -0.019, 0.234, 0.331, 0....","[0.819, -0.072, -0.617, -1.169, -0.67, 0.128, ...","[0.095, 0.315, 0.715, 0.66, 0.735, 0.698, 0.82...","[0.667, 0.674, 0.676, 0.683, 0.694, 0.704, 0.7...","[0.057, 0.059, 0.063, 0.068, 0.074, 0.078, 0.0...","[-0.02, -0.022, -0.027, -0.034, -0.038, -0.04,...","[84, 84, 84, 84, 84, 84, 84, 84, 84, 84, 84, 8..."
1,"Apr 27, 2024 at 11:07:05 AM",Squat,Anwar,46.75,7,225,9,None,16.08,39.98,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[242417, 13, 7, 9, 11, 9, 10, 10, 10, 12, 8, 1...","[0.119, -0.129, -0.155, -0.134, -0.003, 0.076,...","[0.138, -0.047, -0.156, -0.166, -0.181, -0.138...","[0.152, 0.149, 0.097, 0.144, 0.222, 0.199, 0.0...","[0.526, 0.528, 0.529, 0.529, 0.528, 0.527, 0.5...","[0.023, 0.022, 0.022, 0.022, 0.022, 0.022, 0.0...","[-0.006, -0.005, -0.005, -0.005, -0.004, -0.00...","[124, 124, 124, 124, 124, 124, 124, 124, 124, ..."
2,"Mar 2, 2024 at 3:35:29 PM",DB Side Raise,Anwar,32.91,12,27.5,8,None,4.05,30.96,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[95273, 10, 11, 11, 8, 11, 10, 9, 11, 9, 10, 1...","[-0.151, 0.003, -0.038, 0.02, 0.008, -0.123, -...","[0.203, 0.355, 0.324, 0.137, 0.029, -0.009, 0....","[-0.006, -0.054, -0.154, -0.193, -0.148, -0.10...","[0.401, 0.402, 0.403, 0.404, 0.405, 0.405, 0.4...","[0.123, 0.124, 0.124, 0.124, 0.124, 0.123, 0.1...","[-0.025, -0.025, -0.024, -0.023, -0.023, -0.02...","[123, 123, 123, 123, 123, 123, 123, 123, 123, ..."
3,"Apr 25, 2024 at 5:39:23 PM",DB Bench,Anwar,50.13,8,90,9,None,20.37,40,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[211265669, 5, 8, 11, 9, 12, 8, 10, 11, 10, 11...","[0.037, 0.006, -0.063, -0.014, -0.013, -0.023,...","[0.083, 0.018, 0.037, 0.047, 0.015, 0.009, 0.0...","[0.064, 0.07, 0.072, 0.079, 0.063, 0.046, 0.02...","[0.116, 0.117, 0.117, 0.117, 0.116, 0.116, 0.1...","[0.202, 0.202, 0.202, 0.202, 0.201, 0.201, 0.2...","[-0.012, -0.011, -0.011, -0.011, -0.01, -0.01,...","[97, 97, 97, 97, 97, 97, 97, 97, 97, 97, 97, 9..."
4,"Mar 7, 2024 at 7:08:21 AM",Ab Rollout,Anwar,48.09,10,BW,7,None,3.76,42.51,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[90200, 9, 10, 10, 10, 12, 8, 11, 9, 10, 10, 1...","[-0.079, -0.099, -0.128, -0.087, -0.097, 0.003...","[-0.074, -0.097, -0.104, -0.113, -0.133, -0.16...","[0.177, 0.132, 0.157, 0.216, 0.249, 0.254, 0.2...","[0.254, 0.254, 0.255, 0.254, 0.253, 0.252, 0.2...","[0.058, 0.058, 0.058, 0.058, 0.058, 0.058, 0.0...","[-0.007, -0.007, -0.007, -0.006, -0.006, -0.00...","[95, 95, 95, 95, 95, 95, 95, 95, 95, 95, 95, 9..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
197,"Mar 9, 2024 at 4:58:46 PM",DB Deadlift,Anwar,35.30,10,85,7,None,4.75,29.10,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[244571, 13, 4, 10, 10, 10, 10, 10, 12, 7, 12,...","[0.063, 0.147, 0.271, 0.143, 0.202, 0.147, -0....","[0.318, 0.511, 0.429, 0.323, 0.104, -0.049, -0...","[-0.209, -0.099, -0.086, -0.082, -0.008, 0.089...","[0.483, 0.484, 0.486, 0.488, 0.49, 0.49, 0.49,...","[0.055, 0.054, 0.054, 0.053, 0.051, 0.05, 0.04...","[-0.013, -0.014, -0.014, -0.014, -0.013, -0.01...","[116, 116, 116, 116, 116, 116, 116, 116, 116, ..."
198,"Mar 23, 2024 at 11:07:32 AM",DB Bench,Anwar,34.10,10,85,8,None,6.25,23.83,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[10403, 11, 6, 10, 10, 12, 8, 10, 10, 10, 10, ...","[0.052, 0.113, -0.024, 0.064, 0.147, -0.035, -...","[0.001, 0.05, -0.107, -0.043, -0.0, -0.162, -0...","[0.14, 0.121, 0.098, 0.115, 0.133, 0.059, -0.0...","[-0.08, -0.079, -0.08, -0.08, -0.08, -0.081, -...","[-0.147, -0.147, -0.147, -0.147, -0.148, -0.14...","[-0.006, -0.007, -0.007, -0.008, -0.0

### Filter and integrate

In [7]:
from scipy.integrate import cumulative_trapezoid
from scipy.signal import filtfilt, butter

In [8]:
def filter(data, cutoff_freq, sample_rate):
    b, a = butter(1, 2 * cutoff_freq / (sample_rate), btype="highpass")
    return filtfilt(b, a, data)

In [9]:
df['AccX'] = df['AccX'].apply(lambda x: filter(x, CUTOFF_FREQ, SAMPLE_RATE))
df['AccY'] = df['AccY'].apply(lambda x: filter(x, CUTOFF_FREQ, SAMPLE_RATE))
df['AccZ'] = df['AccZ'].apply(lambda x: filter(x, CUTOFF_FREQ, SAMPLE_RATE))

df['VelX'] = df['AccX'].apply(lambda x: cumulative_trapezoid(x, dx=1/SAMPLE_RATE))
df['VelY'] = df['AccY'].apply(lambda x: cumulative_trapezoid(x, dx=1/SAMPLE_RATE))
df['VelZ'] = df['AccZ'].apply(lambda x: cumulative_trapezoid(x, dx=1/SAMPLE_RATE))

df['PosX'] = df['VelX'].apply(lambda x: cumulative_trapezoid(x, dx=1/SAMPLE_RATE))
df['PosY'] = df['VelY'].apply(lambda x: cumulative_trapezoid(x, dx=1/SAMPLE_RATE))
df['PosZ'] = df['VelZ'].apply(lambda x: cumulative_trapezoid(x, dx=1/SAMPLE_RATE))

### Feature engineering

TODO: FFT

In [10]:
from scipy.stats import skew, kurtosis
from scipy.signal import find_peaks
from sklearn.preprocessing import LabelEncoder

In [11]:
feature_map = {
    'Mean': np.mean,
    'Std': np.std,
    'AbsDev': lambda x: np.mean(np.abs(x - np.mean(x))),
    'Median': np.median,
    'MedianAbsDev': lambda x: np.median(np.abs(x - np.median(x))),
    'IQR': lambda x: np.percentile(x, 75) - np.percentile(x, 25),
    'NegCount': lambda x: np.sum(x < 0),
    'PosCount': lambda x: np.sum(x > 0),
    'AboveMeanCount': lambda x: np.sum(x > np.mean(x)),
    'NumPeaks': lambda x: len(find_peaks(x)[0]),
    'Energy': lambda x: np.sum(x ** 2),
    'Skewness': skew,
    'Kurtosis': kurtosis,
    'AvgResultantAcc': lambda x: np.mean(np.sqrt(x ** 2)),
    'SMA': lambda x: np.sum(np.abs(x))
}

## Train an exercise classifier

For each data column

- Extract features from the entire time window for each data set
- Train model to predict exercise based on these feature

### Extract features

In [12]:
feature_dict = {}

for data_column in DATA_COLUMNS:
    for feature_name, feature_func in feature_map.items():
        feature_dict[f'{data_column}{feature_name}'] = df[data_column].apply(feature_func)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['Exercise'])

X = pd.concat(feature_dict, axis=1)
X

,AccXMean,AccXStd,AccXAbsDev,AccXMedian,AccXMedianAbsDev,AccXIQR,AccXNegCount,AccXPosCount,AccXAboveMeanCount,AccXNumPeaks,...,YawIQR,YawNegCount,YawPosCount,YawAboveMeanCount,YawNumPeaks,YawEnergy,YawSkewness,YawKurtosis,YawAvgResultantAcc,YawSMA
0,0.106521,2.237126,1.421469,-0.053526,0.868288,1.762001,1499,1363,1265,459,...,0.2640,133,2728,1947,62,4559.237943,-3.527591,13.381582,1.234496,3533.127
1,0.001938,1.380015,0.807805,-0.080389,0.480538,0.958965,2568,2094,2090,840,...,0.0710,21,4640,3618,179,11264.306547,-3.886383,20.728686,1.538980,7174.723
2,0.045863,1.560772,1.216359,-0.035908,1.115876,2.239310,1666,1616,1561,721,...,0.2230,80,3201,2238,63,9980.173803,-2.570198,7.005121,1.686128,5533.873
3,-0.006109,1.257608,0.830290,-0.008780,0.518401,1.029678,2526,2473,2491,1014,...,0.4605,286,4708,2848,198,3528.968596,0.679907,0.591468,0.695660,3477.602
4,0.008033,1.716912,1.309141,-0.085983,1.022814,2.034769,2527,2269,2258,695,...,0.1175,126,4663,1849,284,2897.219850,2.383175,13.449339,0.728106,3491.994
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
197,-0.003725,1.772077,1.242345,-0.139009,0.871418,1.747591,1912,1609,1614,629,...,0.3670,105,3416,1592,118,9117.578543,-0.976867,2.631069,1.530317,5388.247
198,0.079028,2.325488,1.417666,0.097335,0.868733,1.762063,1595,1802,1711,732,...,0.2160,220,3176,1934,155,6042.830221,-1.240482,5.306523,1.165618,3959.603
199,0.055134,1.679237,1.278564,-0.000590,1.101049,2.186844,1188,1187,1152,493,...,0.2845,36,2336,1722,59,8139.161636,-1.982324,4.005970,1.771590,4207.526
200,-0.015153,1.648597,0.972306,-0.026242,0.561209,1.124053,2394,2228,2275,860,...,0.2330,5,4617,2297,262,13804.525438,0.044775,2.180962,1.641633,7587.630


### Train model

In [13]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

clf = XGBClassifier()
clf.fit(X_train, y_train)

print("Training accuracy: ", clf.score(X_train, y_train))
print("Testing accuracy: ", clf.score(X_test, y_test))

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0))

Training accuracy:  1.0
Testing accuracy:  0.8780487804878049
              precision    recall  f1-score   support

           1       0.90      1.00      0.95         9
           2       0.50      1.00      0.67         3
           3       0.00      0.00      0.00         2
           4       1.00      1.00      1.00         1
           5       0.00      0.00      0.00         1
           6       1.00      1.00      1.00         2
           7       1.00      0.50      0.67         2
           8       0.00      0.00      0.00         1
           9       1.00      1.00      1.00         7
          10       1.00      1.00      1.00         1
          11       1.00      1.00      1.00         3
          12       1.00      1.00      1.00         1
          13       1.00      1.00      1.00         1
          14       0.86      1.00      0.92         6
          15       1.00      1.00      1.00         1

    accuracy                           0.88        41
   macro avg      

## Train a predictor for whether is in rep state or non rep state

 For entries with a RepStartTime and RepEndTime
 - Extract windows for each axes
 - Label window with 0 or 1 depending on whether it is between RepStartTime and RepEndTime
 - Create a classifier for this data

### Extract labelled windows

In [17]:
windowed_data = []

def calculate_sample_index(time, sample_rate):
    return int(float(time)* float(sample_rate))
    
for index, row in df.iterrows():
    for i in range(0, len(row['Counter']) - WINDOW_SIZE, STEP_SIZE):
        start_index = calculate_sample_index(row['StartRepTime'], SAMPLE_RATE)
        end_index = calculate_sample_index(row['EndRepTime'], SAMPLE_RATE)

        windowed_data.append({
            'Exercise': row['Exercise'],
            'Counter': row['Counter'][i:i+WINDOW_SIZE],
            'TimeBetweenSamples': row['TimeBetweenSamples'][i:i+WINDOW_SIZE],
            'AccX': row['AccX'][i:i+WINDOW_SIZE],
            'AccY': row['AccY'][i:i+WINDOW_SIZE],
            'AccZ': row['AccZ'][i:i+WINDOW_SIZE],
            'VelX': row['VelX'][i:i+WINDOW_SIZE],
            'VelY': row['VelY'][i:i+WINDOW_SIZE],
            'VelZ': row['VelZ'][i:i+WINDOW_SIZE],
            'PosX': row['PosX'][i:i+WINDOW_SIZE],
            'PosY': row['PosY'][i:i+WINDOW_SIZE],
            'PosZ': row['PosZ'][i:i+WINDOW_SIZE],
            'Pitch': row['Pitch'][i:i+WINDOW_SIZE],
            'Roll': row['Roll'][i:i+WINDOW_SIZE],
            'Yaw': row['Yaw'][i:i+WINDOW_SIZE],
            'HeartRate': row['HeartRate'][i:i+WINDOW_SIZE],
            'Label': 1 if i >= start_index and i <= end_index else 0
        })

windowed_df = pd.DataFrame(windowed_data)

### Extract features

In [18]:
feature_dict = {}

for data_column in DATA_COLUMNS:
    for feature_name, feature_func in feature_map.items():
        feature_dict[f'{data_column}{feature_name}'] = windowed_df[data_column].apply(feature_func)

y = windowed_df['Label']
X = pd.concat(feature_dict, axis=1)
X

,AccXMean,AccXStd,AccXAbsDev,AccXMedian,AccXMedianAbsDev,AccXIQR,AccXNegCount,AccXPosCount,AccXAboveMeanCount,AccXNumPeaks,...,YawIQR,YawNegCount,YawPosCount,YawAboveMeanCount,YawNumPeaks,YawEnergy,YawSkewness,YawKurtosis,YawAvgResultantAcc,YawSMA
0,0.979700,3.958655,2.857852,0.488519,1.358621,3.068506,39,61,44,7,...,1.03150,26,73,40,1,64.070198,0.713126,-1.053989,0.52794,52.794
1,0.506554,4.733823,3.544734,0.135215,2.413086,4.927193,45,55,38,10,...,0.61800,0,100,67,2,199.493280,-1.176766,0.008951,1.33510,133.510
2,-0.379719,3.684804,2.362995,-0.190097,1.007021,2.699878,54,46,56,11,...,0.09875,0,100,39,4,252.346185,0.322771,-0.781307,1.58675,158.675
3,-0.200113,2.759974,1.754373,0.003187,0.955374,1.987084,50,50,55,11,...,0.17725,0,100,59,3,213.157196,-0.189861,-1.402512,1.45764,145.764
4,-0.098987,0.611882,0.388947,0.003187,0.150531,0.299714,50,50,68,18,...,0.10775,0,100,57,1,204.156558,-0.388618,-1.284520,1.42766,142.766
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13259,-0.856350,6.342411,5.197347,0.253463,4.488139,8.894913,48,52,53,13,...,0.07975,0,100,46,14,246.528717,0.268426,-1.384044,1.56959,156.959
13260,-0.024371,3.981316,3.399045,0.739622,2.976562,6.502183,42,58,58,11,...,0.03550,0,100,20,9,247.818284,1.800181,1.963978,1.57264,157.264
13261,0.875665,2.646408,1.970206,0.778021,1.261124,2.321593,28,72,48,10,...,1.02525,0,100,54,4,153.505948,-0.222601,-1.748780,1.11932,111.932
13262,0.142894,1.520890,1.119704,-0.068128,0.933240,1.646047,52,48,42,9,...,0.31250,0,100,40,2,26.211705,1.772622,3.105023,0.43309,43.309


### Train classifier

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

clf = XGBClassifier()

clf.fit(X_train, y_train)
print("Training accuracy: ", clf.score(X_train, y_train))
print("Testing accuracy: ", clf.score(X_test, y_test))

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0))

Training accuracy:  1.0
Testing accuracy:  0.9536373916321146
              precision    recall  f1-score   support

           0       0.96      0.93      0.95      1225
           1       0.94      0.97      0.96      1428

    accuracy                           0.95      2653
   macro avg       0.95      0.95      0.95      2653
weighted avg       0.95      0.95      0.95      2653

